In [13]:
from core.context import *
from core.checks import *

# Retrieve two sequences, only differing by swapping the positions of i and j.
S1, S2, ctx = get_sequences()

# Left and Right sides of the inequality.
def C(t, lc) -> ArithRef: return W2 * If(t <= lc, RealVal(0), If(t <= lc + 300, ω1 * (t - lc) + ω2, ω3 * (t - lc) + ω4))
lhs = W1 * ((S1.takeoff["i"] - ctx.b["i"]) ** α) + W2 * C(S1.takeoff["i"], ctx.lc["i"]) \
    + W1 * ((S1.takeoff["j"] - ctx.b["j"]) ** α) + W2 * C(S1.takeoff["j"], ctx.lc["j"])
rhs = W1 * ((S2.takeoff["i"] - ctx.b["i"]) ** α) + W2 * C(S2.takeoff["i"], ctx.lc["i"]) \
    + W1 * ((S2.takeoff["j"] - ctx.b["j"]) ** α) + W2 * C(S2.takeoff["j"], ctx.lc["j"])

# Encode and enforce the rule premises; total order on release/tw times and that i and j are δ-equivalent.
premises = [
    ctx.r["i"] <= ctx.r["j"],  ctx.lt["i"] <= ctx.lt["j"],   *ctx.separation_equivalence("i", "j"),
    lhs <= rhs
]

# Encode the claim; no aircraft is more delayed in S1 than S2 (aside i and j).
claim = And(*[
    S1.ctot[x] <= S2.ctot[x]
    for x in ctx.aircraft if x not in ("i", "j")
])

# Check the solver result.
res = verify_pruning_rule(ctx, premises, claim)
print(f"Complete Order CTOT (per-aircraft):  {res}")



# Encode the claim; total delay in S1 ≤ total delay in S2.
claim = Sum([S1.delay[x] for x in ctx.aircraft]) <= Sum([S2.delay[x] for x in ctx.aircraft])

# Check the solver result.
res = verify_pruning_rule(ctx, premises, claim)
print(f"Complete Order Delay (total):        {res}")






Complete Order CTOT (per-aircraft):  Verified
Complete Order Delay (total):        Verified
